# Lab 05 · Streaming events, and the error after the 200
**~20 minutes · costs about $0.01 · Domain 2 API Mechanics, Domain 5 Technical Fundamentals**

Two exam facts: final `usage` and `stop_reason` arrive in **`message_delta`**,
not `message_start`; and because the HTTP status commits before generation
finishes, **an error can arrive after a 200**.

In [ ]:
import os, anthropic
client = anthropic.Anthropic()          # reads ANTHROPIC_API_KEY
MODEL  = "claude-sonnet-4-6"            # verify against lab 00 output
CHEAP  = "claude-haiku-4-5"             # for high-volume steps
print("sdk", anthropic.__version__)

## Print every event in order

Do this once and the sequence sticks.

In [ ]:
from collections import Counter
seen = Counter()

with client.messages.stream(model=MODEL, max_tokens=200,
        messages=[{"role":"user","content":"List three uses for prompt caching."}]) as s:
    for ev in s:
        seen[ev.type] += 1
        if ev.type in ("message_start","content_block_start","content_block_stop",
                       "message_delta","message_stop"):
            print(ev.type)
        if ev.type == "message_delta":
            print("   stop_reason:", ev.delta.stop_reason)
            print("   usage:", ev.usage)

print()
print(dict(seen))

## Confirm where usage lives

`message_start` carries input tokens; the **final** output count only shows up
in `message_delta`. Code that reads usage at the start under-reports cost.

In [ ]:
with client.messages.stream(model=MODEL, max_tokens=150,
        messages=[{"role":"user","content":"Say four sentences about tokens."}]) as s:
    for ev in s:
        if ev.type == "message_start":
            print("at message_start :", ev.message.usage)
        if ev.type == "message_delta":
            print("at message_delta :", ev.usage)
    final = s.get_final_message()
print("final object     :", final.usage, "| stop:", final.stop_reason)

## Partial output is a product decision

If the stream dies at token 300 of 800, the user has already read 300 tokens.
Deleting them is jarring; leaving them is misleading. Decide deliberately.

In [ ]:
import anthropic
buf = []
try:
    with client.messages.stream(model=MODEL, max_tokens=400,
            messages=[{"role":"user","content":"Explain HTTP caching."}]) as s:
        for ev in s:
            if ev.type == "content_block_delta" and ev.delta.type == "text_delta":
                buf.append(ev.delta.text)
except anthropic.APIError as e:
    print("mid-stream failure after a 200:", type(e).__name__)
    print("partial chars already shown to the user:", len("".join(buf)))
    # policy options: mark it truncated, retry silently, or keep and flag
print("ok, chars:", len("".join(buf)))

---
### Checkpoint
- Name the six event types in order
- Where does final `usage` arrive?
- What transport does Claude streaming use? (Not WebSocket.)